In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import random
from random import random as rd
import gc

from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import HistGradientBoostingRegressor  # stronger non‑linear model



In [2]:
dtype_train = {
    "id": "int32",
    "breath_id": "int32",
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "int8",
    "pressure": "float32",
}
df_train = pd.read_csv(
    "../input/ventilator-pressure-prediction/train.csv",
    dtype=dtype_train,
    usecols=list(dtype_train.keys()),
)

# Compute pressure bounds early and free the column later to reduce memory pressure
min_p, max_p = df_train["pressure"].min(), df_train["pressure"].max()


def set_seed(seed=2021):
    np.random.seed(seed)
    random_state = np.random.RandomState(seed)
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    return random_state


def add_engineered_features(df):
    """Create additional features that capture intra‑breath dynamics, cheap interactions,
    and simple lag information. This version avoids converting breath_id to categorical.
    """
    df = df.copy()

    # Keep breath_id as int32 for fast groupby operations
    df["breath_id"] = df["breath_id"].astype("int32")

    cum = df.groupby("breath_id", sort=False)[["u_in", "time_step"]].cumsum()
    df["cum_u_in"] = cum["u_in"]
    df["cum_time"] = cum["time_step"]

    shifted = df.groupby("breath_id", sort=False)[["u_in", "u_out", "time_step"]].shift(
        1
    )
    df["prev_u_in"] = shifted["u_in"].fillna(0)
    df["prev_u_out"] = shifted["u_out"].fillna(0)
    df["delta_time"] = shifted["time_step"].fillna(0)

    df["u_in_sq"] = df["u_in"] * df["u_in"]
    df["time_sq"] = df["time_step"] * df["time_step"]
    df["R_C"] = df["R"] * df["C"]
    df["u_in_u_out"] = df["u_in"] * df["u_out"]

    engineered_cols = [
        "cum_u_in",
        "cum_time",
        "prev_u_in",
        "prev_u_out",
        "delta_time",
        "u_in_sq",
        "time_sq",
        "R_C",
        "u_in_u_out",
    ]
    for col in engineered_cols:
        df[col] = df[col].astype(np.float32)

    # Ensure breath_id remains int32 for downstream use
    df["breath_id"] = df["breath_id"].astype("int32")
    return df




In [3]:
def generate_submission():
    """Create a submission using a HistGradientBoostingRegressor with engineered columns,
    without unnecessary high‑cardinality features and with a stronger model capacity."""
    dtype_test = {
        "id": "int32",
        "breath_id": "int32",
        "R": "int16",
        "C": "int16",
        "time_step": "float32",
        "u_in": "float32",
        "u_out": "int8",
    }
    df_test = pd.read_csv(
        "../input/ventilator-pressure-prediction/test.csv",
        dtype=dtype_test,
        usecols=list(dtype_test.keys()),
    )

    # Engineer features for train and test
    train_fe = add_engineered_features(df_train)
    test_fe = add_engineered_features(df_test)

    # Free the original raw training dataframe early
    del df_train
    gc.collect()

    base_features = ["R", "C", "u_in", "u_out", "time_step"]
    engineered = [
        "cum_u_in",
        "cum_time",
        "u_in_sq",
        "time_sq",
        "R_C",
        "u_in_u_out",
        "prev_u_in",
        "prev_u_out",
        "delta_time",
    ]
    feature_cols = base_features + engineered

    X_train = train_fe[
        feature_cols
    ].values  # already float32 from add_engineered_features
    X_test = test_fe[feature_cols].values

    y_train = train_fe["pressure"].values.astype(np.float32)

    model = HistGradientBoostingRegressor(
        max_iter=5000,
        learning_rate=0.01,
        max_depth=12,
        validation_fraction=0.2,
        n_iter_no_change=20,
        random_state=2021,
        verbose=0,
    )
    model.fit(X_train, y_train)

    # Release memory not needed for prediction
    del X_train, y_train, train_fe, df_test, test_fe
    gc.collect()

    preds = model.predict(X_test)

    # Clip predictions to the original pressure range
    preds = np.clip(preds, min_p, max_p)

    submission = pd.DataFrame({"id": df_test["id"], "pressure": preds})
    submission.to_csv("submission.csv", index=False)
    print("Submission written to submission.csv with shape:", submission.shape)




In [4]:
generate_submission()

UnboundLocalError: cannot access local variable 'df_train' where it is not associated with a value